# OSU AI Club Poker Competition 2026
*A Guide to Building and Submitting Your Poker Agent*

---

### Table of Contents

1. [Installation Guide](#installation-guide)
2. [Poker Environment Overview](#poker-environment-overview)
3. [Create an Agent](#create-an-agent)
4. [Test Your Agent](#test-your-agent)
5. [Competition Rules & Submission](#competition-rules-and-rewards)


## Installation Guide

Before you can start creating an agent, you must install the necessary dependicies. From the project root directory, run:

```
pip install -r requirements.txt
```

**Optional:** For a clean environment, consider using a virtual environment first:

```
python -m venv venv
```

Then activate it (e.g., `venv\Scripts\activate` on Windows, `source venv/bin/activate` on macOS/Linux) and run the `pip install` command above.

## Poker Environment Overview

This competition utilizes a custom implementation of Petting Zoo's [Texas Hold'em No Limit Environment](https://pettingzoo.farama.org/environments/classic/texas_holdem_no_limit/).

### Observation Space

In this custom implementation, the observation is a dictionary containing three elements: the original `observation` and `action_mask` included in Petting Zoo's Environment, and a third `human_readable` element designed to support rule-based bots.

#### 1. The Reinforcement Learning Vector
> **Key:** `observation['observation']`

This element matches the original Petting Zoo implementation and is ideal for **Reinforcement Learning** models.
* **Structure:** A 54-length vector.
* **Content:** The first 52 entries represent the cards in the player's hand and community cards.
    * `1`: The card is present (in hand or on board).
    * `0`: The card is not present.

![rl-observation-space](<images/rl-observation-space.png>)
---

#### 2. Human-Readable State
> **Key:** `observation['human_readable']`

This is a custom dictionary designed for **rule-based decision making** and debugging. It translates the raw vector into explicit values (e.g., current pot, specific card ranks, stack sizes).

![human-readable-observation-space](<images/human-readable-observation-space.png>)
---

#### 3. Action Mask
> **Key:** `observation['action_mask']`

This element tells the agent what moves it can make, given the gamestate.
* **Structure:** A binary vector with **5 entries**.
* **Function:** Each entry corresponds to a specific action.
    * `1` = Action is **Legal**.
    * `0` = Action is **Illegal**.

![action-mask](<images/action-space.png>)

## Create an Agent

An agent is a simple Python class responsible for making decisions. It must implement a method called `act` which receives the current game state (`observation`) and the game environment (`env`). Your goal is to use the observation data to return a valid integer action.

### Agent Structure

You will design a Python class with the following structure:

In [ ]:
import numpy as np 
from typing import Dict
from texas_holdem_env import TexasHoldEm, Action

class AlwaysFoldAgent:
    def act(self, observation: Dict, env: TexasHoldEm) -> int:
        """
        Determines the agent's action for the current turn.

        Args:
            observation (dict): A dictionary containing:
                - 'observation': The raw RL vector.
                - 'human_readable': Parsed game state (cards, pot, etc.).
                - 'action_mask': Binary vector indicating valid moves.
            env (TexasHoldEm): The game environment instance, used to 
                               access constants like env.FOLD or env.CHECK.

        Returns:
            int: The index of the action to take.
        """
        # Get the action mask (binary vector of valid moves)
        # Example: action_mask = [1, 0, 1, 1, 0], indicates actions 0, 2, and 3 are valid.
        action_mask = observation["action_mask"]

        # Get the integer indices of all legal actions
        # Example: turns [1, 0, 1, 1, 0] -> [0, 2, 3]
        valid_actions = np.flatnonzero(action_mask)  

        # Always try to Fold if it's legal.
        # If not, pick the first available legal action  
        # NOTE: Action.FOLD is an ENUM and equivalent to 0 
        if Action.FOLD in valid_actions:
            return Action.FOLD
        return valid_actions[0]

## Test Your Agent

Once you have implemented your agent's logic, you will need to validate its performance. We have provided a `test_agent.py` file for this purpose. 

You can test your agent in two ways:
1. **Debugging (`run_hand`)**: Simulates a single hand with detailed logs. Use this to verify your agent doesn't crash and behaves logically. (NOTE: THIS DOESN'T ACTUALLY PRINT DETAILED LOGS YET)
2. **Benchmarking (`run_competition`)**: Simulates thousands of hands to determine win rates and average chip gains.

### How to Run Tests

Open `test_agent.py` and import your agent class. Then, instantiate your agent and an opponent (e.g., the `RandomAgent` or another custom agent) and pass them to the simulation functions.

In [ ]:
from texas_holdem_env import TexasHoldEm
from examples.example_usage import run_competition, run_hand
from examples.example_agents import AlwaysFoldAgent, RandomAgent

# TODO: Import your agent from workspace and use it below
# from workspace.agent import MyAgent


def main():
    # Run a full competition
    # Arguments: (Number of Hands, Player 1 Agent, Player 2 Agent)
    # This will print the win-rates and stats to the console.
    run_competition(10000, AlwaysFoldAgent(), RandomAgent())

    # Run a single debug hand
    run_hand(RandomAgent(), AlwaysFoldAgent())

if __name__ == "__main__":
    main()

## Competition Rules and Rewards

TODO: THESE RULES AND REWARDS NEED TO BE DISCUSSED AND UPDATED AND ALL CURRENT RULES ARE SIMPLY PLACEHOLDERS. NOT SURE ABOUT TIME/MEM REQUIREMENTS, USE OF OUTSIDE RESOURCES (IDEALLY PEOPLE CAN'T JUST COPY PASTE SOMEONE ELSE'S CODE), AND PRETTY MUCH ALL OF IT.

### Tournament Structure
The competition will be structured as a **1v1 Tournament**. 

* **Format:** [TODO: ROUND ROBIN / SINGLE ELIMINATION BRACKET]
* **Match Length:** Each match consists of **10,000 hands**.
* **Fairness (Duplicate Poker):** To reduce luck, we will use a "Duplicate Poker" system. 
    * Agents will play 5,000 hands as Player 1 and 5,000 hands as Player 2.
    * The same RNG seeds will be used for both sets, meaning both agents will face the exact same card distribution. The agent that accumulates the most chips across both legs wins the match.

### Submission Requirements
1.  **Language:** All agents must be written in **Python**.
2.  **File Format:** Submit a single file named `agent.py` (or `[TeamName]_agent.py`).
4.  **Dependencies:** You may use standard Python libraries (math, random, etc.) and `numpy`.
    * *Prohibited:* [TODO: NOT SURE IF WE WANT TO PROHIBIT ANY LIBRARIES OR ]

### Agent Constraints & Fair Play
To ensure the tournament runs smoothly, all agents must adhere to the following strict constraints:

1.  **Time Limit:** Your `act` function must return a move within **[0.1] seconds**. Agents exceeding this limit will strictly Fold or be disqualified.
2.  **Memory Limit:** Your agent must not exceed **[500 MB]** of RAM.
3.  **Statelessness:** Agents should not attempt to store data between matches (e.g., writing to disk is prohibited).
4.  **No Cheating:** * Agents must not attempt to access the `env` object's internal variables (e.g., looking at the opponent's cards or the deck).
    * Agents must not attempt to access the internet.
    * Any attempt to "hack" the runner script will result in immediate disqualification.
    * The use of AI is allowed and outside resources are allowed. NOTE: UPDATE THIS LINE. We reccommend not copying code from others?

### Rewards
Prizes will be awarded to the top performing agents:

* 🥇 **1st Place:** [TODO: PRIZE, e.g., $50 Gift Card and AIC Merchandise]
* 🥈 **2nd Place:** [TODO: PRIZE $25 Gift Card]
* 🥉 **3rd Place:** [TODO: PRIZE $25 Gift Card]

---
**How to Submit:**
Please email your `agent.py` file to **sullivk3@oregonstate.edu** by **END OF SPRING Week 10 (June something)**.